In [1]:
# !pip install psutil pandas openpyxl
import csv
import datetime
import glob
import json
import os
import platform
import re
import socket
import subprocess

import pandas as pd
import psutil
from openpyxl import load_workbook
from openpyxl.styles import Font

GB = 1024 ** 3     # 1 GB = 1024^3 bytes, bij iedereen hetzelfde

## 1 - Basis Script uitvoeren

- Voer het script collect_sysinfo.py uit.
- Bekijk aandachtig de structuur van de dictionary info.
- Identificeer welke keys en geneste keys relevante informatie bevatten, bijvoorbeeld CPU, RAM-geheugen, opslag, netwerkinterfaces, computernaam en model.

In [2]:
now = datetime.datetime.now()
print(now)

def collect_system_info():
    data = {}

    
    data["cpu"] = {
        "platform": platform.platform(),
        "processor": platform.processor(),
        "physical_cores": psutil.cpu_count(logical=False),
        "logical_processors": psutil.cpu_count(logical=True),
        "cpu_freq": psutil.cpu_freq()._asdict() if psutil.cpu_freq() else None,
        "cpu_percent_total": psutil.cpu_percent(interval=0.1),
        "cpu_percent_per_cpu": psutil.cpu_percent(interval=0.1, percpu=True),
    }

    
    data["memory"] = psutil.virtual_memory()._asdict()
    data["swap"] = psutil.swap_memory()._asdict()

    
    storage = {}
    for part in psutil.disk_partitions(all=False):
        try:
            usage = psutil.disk_usage(part.mountpoint)
            storage[part.mountpoint] = {
                "device": part.device,
                "fstype": part.fstype,
                "opts": part.opts,
                "total": usage.total,
                "used": usage.used,
                "free": usage.free,
                "percent": usage.percent,
            }
        except PermissionError:
            continue
    data["storage"] = storage

    
    net = {}
    addrs = psutil.net_if_addrs()
    stats = psutil.net_if_stats()
    io = psutil.net_io_counters(pernic=True)
    for iface, addr_list in addrs.items():
        net[iface] = {
            "addrs": [a._asdict() for a in addr_list],
            "stats": stats[iface]._asdict() if iface in stats else None,
            "io": io[iface]._asdict() if iface in io else None,
        }

    data["network"] = {
        "hostname": socket.gethostname(),
        "fqdn": socket.getfqdn(),
        "interfaces": net,
    }

    return data

info = collect_system_info()
print(json.dumps(info, indent=2))

2026-10-14 02:39:26.475646
{
  "cpu": {
    "platform": "Windows-11-10.0.26200-SP0",
    "processor": "AMD64 Family 25 Model 80 Stepping 0, AuthenticAMD",
    "physical_cores": 3,
    "logical_processors": 5,
    "cpu_freq": {
      "current": 2301.0,
      "min": 0.0,
      "max": 2301.0
    },
    "cpu_percent_total": 23.3,
    "cpu_percent_per_cpu": [
      0.0,
      0.0,
      0.0,
      0.0,
      0.0
    ]
  },
  "memory": {
    "total": 7852855296,
    "available": 1201999872,
    "percent": 84.7,
    "used": 6650855424,
    "free": 1201999872
  },
  "swap": {
    "total": 12025974784,
    "used": 741412864,
    "free": 11284561920,
    "percent": 6.2,
    "sin": 0,
    "sout": 0
  },
  "storage": {
    "C:\\": {
      "device": "C:\\",
      "fstype": "NTFS",
      "opts": "rw,fixed",
      "total": 253507211264,
      "used": 197362499584,
      "free": 56144711680,
      "percent": 77.9
    }
  },
  "network": {
    "hostname": "Unluckydodo",
    "fqdn": "Unluckydodo.home",


In [3]:

print("Hoofdsleutels:    ", list(info.keys()))
print("Sleutels cpu:     ", list(info["cpu"].keys()))
print("Sleutels memory:  ", list(info["memory"].keys()))
print("Schijven:         ", list(info["storage"].keys()))
print("Sleutels network: ", list(info["network"].keys()))
print("Interfaces:")
for naam, d in info["network"]["interfaces"].items():
    up = d["stats"]["isup"] if d["stats"] else None
    ipv4 = [a["address"] for a in d["addrs"] if a["family"] == socket.AF_INET]
    print(f"   {naam:35} up={up}  ipv4={ipv4}")

Hoofdsleutels:     ['cpu', 'memory', 'swap', 'storage', 'network']
Sleutels cpu:      ['platform', 'processor', 'physical_cores', 'logical_processors', 'cpu_freq', 'cpu_percent_total', 'cpu_percent_per_cpu']
Sleutels memory:   ['total', 'available', 'percent', 'used', 'free']
Schijven:          ['C:\\']
Sleutels network:  ['hostname', 'fqdn', 'interfaces']
Interfaces:
   Ethernet 2                          up=True  ipv4=['172.16.150.1']
   LAN-verbinding* 1                   up=False  ipv4=['169.254.167.228']
   LAN-verbinding* 2                   up=False  ipv4=['169.254.102.185']
   Wi-Fi                               up=True  ipv4=['192.168.1.38']
   Ethernet                            up=False  ipv4=['169.254.145.136']
   Loopback Pseudo-Interface 1         up=True  ipv4=['127.0.0.1']


**Waar zit welke informatie?**

| Gegeven | Pad in de dictionary | Opmerking |
|---|---|---|
| Computernaam | `info["network"]["hostname"]` | |
| Computermodel | *niet aanwezig* | moet apart opgevraagd worden (opdracht 2) |
| Serienummer | *niet aanwezig* | moet apart opgevraagd worden (opdracht 8) |
| Processor | `info["cpu"]["processor"]` | |
| Fysieke cores | `info["cpu"]["physical_cores"]` | |
| Max. CPU-frequentie | `info["cpu"]["cpu_freq"]["max"]` | in MHz, kan `None` zijn |
| RAM | `info["memory"]["total"]` | in bytes |
| Bestandssysteem | `info["storage"][schijf]["fstype"]` | per schijf/partitie |
| Opslagcapaciteit | `info["storage"][schijf]["total"]` | in bytes |
| MAC-adres | `info["network"]["interfaces"][naam]["addrs"]` | element met `family == -1` (`psutil.AF_LINK`) |
| IPv4 | idem | `family == 2` |
| IPv6 | idem | `family == 23` (`fe80::` = link-local) |

## 2 - Informatie filteren
Schrijf een script dat uit de verzamelde systeeminformatie de gevraagde gegevens selecteert en in een nieuwe Python dictionary opslaat:
computernaam, computermodel, processor, aantal fysieke CPU-cores, maximale CPU-frequentie, totaal RAM, bestandssysteem, opslagcapaciteit, MAC-adres, IPv4, IPv6 en IPv6 link-local van de primaire netwerkinterface. Daarbij komt het **serienummer** (opdracht 8).

### Gekozen criteria (eerst beschreven, daarna geïmplementeerd)

**Opslag.** Een computer kan meerdere schijven en partities hebben. Ik kies de **systeemschijf** (`C:\` op Windows, `/` op Linux en macOS). Motivatie: dat is de schijf waarop het besturingssysteem staat, die bestaat op elke computer en dus is de dataset tussen groepsleden vergelijkbaar. Externe schijven en USB-sticks verschillen per computer en per moment.

**Primaire netwerkinterface.** Criteria, in volgorde:
1. De interface waarlangs het verkeer naar het internet gaat. Het script maakt een UDP-socket naar `8.8.8.8` (er wordt niets verstuurd) en leest het lokale IP af. De interface met dat IP is de primaire.
2. Als dat mislukt (geen internet): de eerste interface die **up** is en een IPv4-adres heeft dat **niet** met `127.` (loopback) of `169.254` (APIPA, geen echte verbinding) begint.

Motivatie: virtuele adapters (VirtualBox, VPN), loopback en niet-verbonden adapters worden zo overgeslagen, en de regel werkt op Windows, Linux en macOS.

**Eenheden.** RAM en opslag in **GB** (1 GB = 1024³ bytes, afgerond op 2 decimalen), CPU-frequentie in **MHz**. Dezelfde eenheden voor alle groepsleden.

**Ontbrekende gegevens.** Computermodel en serienummer zitten niet in psutil. Die vraag ik apart op bij het besturingssysteem (PowerShell op Windows, `sysctl`/`ioreg` op macOS, `/sys/class/dmi` op Linux). Is er niets bruikbaar, dan wordt de waarde `N/A`.

In [4]:
KOLOMMEN = ["hostname", "model", "serial_number", "processor", "physical_cores",
            "cpu_max_mhz", "ram_gb", "fs_type", "storage_gb", "primary_iface",
            "mac", "ipv4", "ipv6", "ipv6_link_local"]

# Teksten die fabrikanten invullen als er geen echt serienummer is
ONGELDIGE_SERIALS = {"", "0", "none", "default string", "to be filled by o.e.m.",
                     "system serial number", "not specified", "n/a", "unknown",
                     "o.e.m.", "to be filled by oem", "not applicable"}

def _run(cmd, timeout=15):
    try:
        out = subprocess.run(cmd, capture_output=True, text=True, timeout=timeout)
        return out.stdout.strip()
    except Exception:
        return ""

def _ps(cmd):
    return _run(["powershell", "-NoProfile", "-Command", cmd], timeout=30)

def get_model():
    """Computermodel zit niet in psutil."""
    system = platform.system()
    if system == "Windows":
        out = _ps("(Get-CimInstance Win32_ComputerSystem).Model")
    elif system == "Darwin":
        out = _run(["sysctl", "-n", "hw.model"])
    else:
        out = _run(["cat", "/sys/devices/virtual/dmi/id/product_name"])
    return out or "N/A"

def get_serial():
    """Serienummer (opdracht 8) zit niet in psutil."""
    system = platform.system()
    serial = ""
    if system == "Windows":
        serial = _ps("(Get-CimInstance Win32_BIOS).SerialNumber")
    elif system == "Darwin":
        out = _run(["ioreg", "-rd1", "-c", "IOPlatformExpertDevice"])
        m = re.search(r'"IOPlatformSerialNumber"\s*=\s*"([^"]+)"', out)
        serial = m.group(1) if m else ""
    else:
        for pad in ("/sys/class/dmi/id/product_serial",
                    "/sys/class/dmi/id/board_serial",
                    "/sys/class/dmi/id/chassis_serial"):
            try:
                with open(pad) as f:
                    serial = f.read().strip()
            except OSError:
                serial = ""
            if serial.lower() not in ONGELDIGE_SERIALS:
                break
    return serial if serial.lower() not in ONGELDIGE_SERIALS else "N/A"

info["system"] = {"model": get_model(), "serial_number": get_serial()}
print(info["system"])

{'model': 'HP ProBook 455 G8 Notebook PC', 'serial_number': '5CD111NW3W'}


In [5]:
def get_primary_interface(net):
    """Interface waarlangs het verkeer naar het internet gaat."""
    try:
        s = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
        s.connect(("8.8.8.8", 80))   
        local_ip = s.getsockname()[0]
        s.close()
        for name, data in net["interfaces"].items():
            if any(a["address"] == local_ip for a in data["addrs"]):
                return name, data
    except Exception:
        pass

    for name, data in net["interfaces"].items():
        if not (data["stats"] and data["stats"]["isup"]):
            continue
        for a in data["addrs"]:
            if a["family"] == socket.AF_INET and not a["address"].startswith(("169.254", "127.")):
                return name, data
    return None, None


def filter_info(info):
    net = info["network"]
    name, iface = get_primary_interface(net)

    mac = ipv4 = ipv6 = link_local = "N/A"
    if iface:
        for a in iface["addrs"]:
            addr = a["address"].split("%")[0]
            if a["family"] == psutil.AF_LINK:
                mac = addr
            elif a["family"] == socket.AF_INET:
                ipv4 = addr
            elif a["family"] == socket.AF_INET6:
                if addr.lower().startswith("fe80"):
                    link_local = addr
                elif ipv6 == "N/A":
                    ipv6 = addr

    system_drive = "C:\\" if platform.system() == "Windows" else "/"
    disk = info["storage"].get(system_drive) or next(iter(info["storage"].values()), {})
    freq = psutil.cpu_freq()

    return {
        "hostname":        net["hostname"],
        "model":           info["system"]["model"],
        "serial_number":   info["system"]["serial_number"],
        "processor":       info["cpu"]["processor"],
        "physical_cores":  info["cpu"]["physical_cores"],
        "cpu_max_mhz":     round(freq.max, 0) if freq else "N/A",
        "ram_gb":          round(info["memory"]["total"] / GB, 2),
        "fs_type":         disk.get("fstype", "N/A"),
        "storage_gb":      round(disk.get("total", 0) / GB, 2),
        "primary_iface":   name or "N/A",
        "mac":             mac,
        "ipv4":            ipv4,
        "ipv6":            ipv6,
        "ipv6_link_local": link_local,
    }


filtered = filter_info(info)
for k, v in filtered.items():
    print(f"{k:16} {v}")

hostname         Unluckydodo
model            HP ProBook 455 G8 Notebook PC
serial_number    5CD111NW3W
processor        AMD64 Family 25 Model 80 Stepping 0, AuthenticAMD
physical_cores   3
cpu_max_mhz      2301.0
ram_gb           7.31
fs_type          NTFS
storage_gb       236.1
primary_iface    Wi-Fi
mac              70-9C-D1-E9-BE-BB
ipv4             192.168.1.38
ipv6             N/A
ipv6_link_local  N/A


## 3 - JSON
- Zet je gefilterde dictionary om naar JSON.
- Bewaar de gegevens in een .json-bestand.
- Lees het bestand opnieuw in als een Python dictionary.
- Controleer na het opnieuw inlezen of de belangrijkste gegevens overeenkomen met de oorspronkelijke dictionary.
- Leg in enkele zinnen concreet uit wat het verschil is tussen een Python dict en JSON.

In [6]:
json_tekst = json.dumps(filtered, indent=2)
print(type(filtered), "->", type(json_tekst))
print(json_tekst)

bestandsnaam = f"sysinfo_{filtered['hostname']}.json"
with open(bestandsnaam, "w", encoding="utf-8") as f:
    json.dump(filtered, f, indent=2)
print("Opgeslagen als:", bestandsnaam)

with open(bestandsnaam, "r", encoding="utf-8") as f:
    ingelezen = json.load(f)
print(type(ingelezen))

print("Zijn ze gelijk?", ingelezen == filtered)
for sleutel in ["hostname", "model", "serial_number", "ram_gb", "ipv4", "mac"]:
    print(f"{sleutel:14} {filtered[sleutel]}  |  {ingelezen[sleutel]}")

<class 'dict'> -> <class 'str'>
{
  "hostname": "Unluckydodo",
  "model": "HP ProBook 455 G8 Notebook PC",
  "serial_number": "5CD111NW3W",
  "processor": "AMD64 Family 25 Model 80 Stepping 0, AuthenticAMD",
  "physical_cores": 3,
  "cpu_max_mhz": 2301.0,
  "ram_gb": 7.31,
  "fs_type": "NTFS",
  "storage_gb": 236.1,
  "primary_iface": "Wi-Fi",
  "mac": "70-9C-D1-E9-BE-BB",
  "ipv4": "192.168.1.38",
  "ipv6": "N/A",
  "ipv6_link_local": "N/A"
}
Opgeslagen als: sysinfo_Unluckydodo.json
<class 'dict'>
Zijn ze gelijk? True
hostname       Unluckydodo  |  Unluckydodo
model          HP ProBook 455 G8 Notebook PC  |  HP ProBook 455 G8 Notebook PC
serial_number  5CD111NW3W  |  5CD111NW3W
ram_gb         7.31  |  7.31
ipv4           192.168.1.38  |  192.168.1.38
mac            70-9C-D1-E9-BE-BB  |  70-9C-D1-E9-BE-BB


**Verschil tussen een Python dict en JSON**

Een **dict** is een Python-object in het geheugen van je programma. Je kan er direct mee werken (`d["hostname"]`) en het kan Python-specifieke waarden bevatten zoals `True`, `None` en tuples. **JSON** is een tekstformaat (een `str`) om gegevens op te slaan of te versturen naar andere programma's en talen. Het lijkt op een dict, maar de regels zijn strenger: sleutels staan altijd tussen dubbele aanhalingstekens en `True`/`None` worden `true`/`null`. `json.dumps()` zet een dict om naar JSON-tekst en `json.loads()` zet die tekst terug om naar een dict. Een dict gebruik je dus binnen Python, JSON voor opslag en uitwisseling, bijvoorbeeld in API's.

## 4 - Transformeren van informatie naar CSV
De gefilterde informatie zit in een Python dict. Hier transformeren we ze naar een CSV-bestand dat in een spreadsheetprogramma kan worden geopend.

**Gemaakte keuzes**
- **Bestandsnaam:** `sysinfo_<hostname>.csv`. De hostname maakt elk bestand uniek, zodat bij het samenvoegen (opdracht 6) niets overschreven wordt.
- **Kolommen:** de 14 sleutels van de gefilterde dict (zie `KOLOMMEN`), in een vaste volgorde, met Engelstalige, korte namen zonder spaties.
- **Één computer = één rij**, met de kolomnamen op de eerste regel.
- **Scheidingsteken:** `;`. In Belgische/Nederlandse instellingen gebruikt Excel `;` omdat de komma het decimaalteken is.
- **Encoding:** `utf-8-sig`, zodat Excel speciale tekens correct toont.
- **Ontbrekende informatie:** `N/A`, nooit een lege cel, zodat een ontbrekende waarde herkenbaar is.
- **Meerdere exemplaren** (schijven, interfaces): daarom is gekozen voor de systeemschijf en de primaire interface. Zo blijft er per computer exact één waarde per kolom over.

In [7]:
csv_naam = f"sysinfo_{filtered['hostname']}.csv"

with open(csv_naam, "w", newline="", encoding="utf-8-sig") as f:
    writer = csv.DictWriter(f, fieldnames=KOLOMMEN, delimiter=";")
    writer.writeheader()
    writer.writerow({k: filtered.get(k, "N/A") for k in KOLOMMEN})
print("Opgeslagen als:", csv_naam)

with open(csv_naam, "r", newline="", encoding="utf-8-sig") as f:
    for rij in csv.DictReader(f, delimiter=";"):
        for k, v in rij.items():
            print(f"{k:16} {v}")

Opgeslagen als: sysinfo_Unluckydodo.csv
hostname         Unluckydodo
model            HP ProBook 455 G8 Notebook PC
serial_number    5CD111NW3W
processor        AMD64 Family 25 Model 80 Stepping 0, AuthenticAMD
physical_cores   3
cpu_max_mhz      2301.0
ram_gb           7.31
fs_type          NTFS
storage_gb       236.1
primary_iface    Wi-Fi
mac              70-9C-D1-E9-BE-BB
ipv4             192.168.1.38
ipv6             N/A
ipv6_link_local  N/A


## 5 - Transformeren naar een spreadsheetbestand

- **Formaat:** `.xlsx` (Excel).
- **Library:** `openpyxl` voor opmaak en `pandas` om het CSV-bestand in te lezen en weg te schrijven.
- Dezelfde gegevens en kolomnamen als in het CSV-bestand. Het serienummer lezen we als **tekst** in, zodat voorloopnullen niet verdwijnen.

In [8]:
xlsx_naam = f"sysinfo_{filtered['hostname']}.xlsx"

df = pd.read_csv(csv_naam, sep=";", encoding="utf-8-sig", keep_default_na=False,
                 dtype={"serial_number": str})
df.to_excel(xlsx_naam, index=False, sheet_name="sysinfo")

wb = load_workbook(xlsx_naam)
ws = wb["sysinfo"]
for cel in ws[1]:
    cel.font = Font(bold=True)
for kolom in ws.columns:
    breedte = max(len(str(c.value)) for c in kolom if c.value is not None)
    ws.column_dimensions[kolom[0].column_letter].width = breedte + 2
ws.freeze_panes = "A2"
wb.save(xlsx_naam)
print("Opgeslagen als:", xlsx_naam)

df_terug = pd.read_excel(xlsx_naam, sheet_name="sysinfo", keep_default_na=False,
                         dtype={"serial_number": str})
ws = load_workbook(xlsx_naam)["sysinfo"]
print("Werkblad:         ", ws.title)
print("Aantal kolommen:  ", ws.max_column)
print("Aantal rijen:     ", ws.max_row, "(1 kopregel + gegevens)")
print("Kopregel vet:     ", ws["A1"].font.bold)
print("Bovenste rij vast:", ws.freeze_panes)
try:
    pd.testing.assert_frame_equal(df_terug, df, check_dtype=False)
    print("Gelijk aan CSV: True")
except AssertionError as e:
    print("Gelijk aan CSV: False")
    print(e)
df_terug.T

Opgeslagen als: sysinfo_Unluckydodo.xlsx
Werkblad:          sysinfo
Aantal kolommen:   14
Aantal rijen:      2 (1 kopregel + gegevens)
Kopregel vet:      True
Bovenste rij vast: A2
Gelijk aan CSV: True


,0
hostname,Unluckydodo
model,HP ProBook 455 G8 Notebook PC
serial_number,5CD111NW3W
processor,"AMD64 Family 25 Model 80 Stepping 0, AuthenticAMD"
physical_cores,3
cpu_max_mhz,2301
ram_gb,7.31
fs_type,NTFS
storage_gb,236.1
primary_iface,Wi-Fi


**CSV tegenover een echt spreadsheetbestand**

Een **CSV-bestand** is gewone tekst: één regel per rij, waarden gescheiden door `;`. Het bevat alleen gegevens, geen opmaak, geen formules en geen meerdere werkbladen. Een **.xlsx-bestand** is een gestructureerd (gezipt XML-)bestand met naast de gegevens ook celtypes (getal, tekst, datum), opmaak (vet, kleuren, kolombreedte), vastgezette rijen, formules, grafieken en meerdere werkbladen.

**Wanneer CSV?** Voor uitwisseling tussen programma's en scripts, voor grote datasets en wanneer het bestand klein en leesbaar moet blijven (bijvoorbeeld op GitHub of in een pipeline). **Wanneer .xlsx?** Wanneer mensen het bestand lezen of presenteren en je opmaak, meerdere werkbladen of formules wilt. Daarom gebruiken we CSV om de bestanden van de groep samen te voegen, en .xlsx voor het leesbare eindresultaat.

## 6 - Informatie van verschillende computers samenvoegen
Ieder groepslid heeft een bestand met de informatie over zijn of haar computer. Met Python voegen we die samen in één CSV.

**Afspraken binnen het team** (aan te passen aan wat jullie echt afspreken)
- **Kolommen:** exact de 14 kolommen van `KOLOMMEN`, inclusief `serial_number`, met `;` als scheidingsteken en `utf-8-sig`.
- **Bestandsnaam:** `sysinfo_<hostname>.csv`.
- **Uitwisselen:** alle bestanden worden in één gedeelde map (bijvoorbeeld een GitHub-repo of Teams-map) geplaatst.

**Hoe het script werkt**
- **Welke bestanden:** het script zoekt zelf alle `sysinfo_*.csv`-bestanden in de map (met `glob`) en slaat `sysinfo_all.csv` over. Een nieuwe computer toevoegen = zijn CSV in de map zetten en het script opnieuw draaien.
- **Verschillende kolomnamen:** elk bestand wordt tot dezelfde 14 kolommen in dezelfde volgorde teruggebracht.
- **Ontbrekende gegevens:** ontbreekt een kolom (bijvoorbeeld `serial_number` in een oud bestand) of is een cel leeg, dan wordt die `N/A`, en er wordt een waarschuwing getoond.
- **Dubbels:** één rij per computer (op `hostname`). Wordt een bestand opnieuw ingeleverd, dan blijft de laatste versie staan.

In [9]:
map_met_csv = "."
bestanden = sorted(glob.glob(os.path.join(map_met_csv, "sysinfo_*.csv")))
bestanden = [b for b in bestanden if os.path.basename(b) != "sysinfo_all.csv"]
print("Gevonden bestanden:")
for b in bestanden:
    print("  ", os.path.basename(b))

frames = []
for b in bestanden:
    d = pd.read_csv(b, sep=";", encoding="utf-8-sig", keep_default_na=False, dtype=str)
    ontbreekt = [k for k in KOLOMMEN if k not in d.columns]
    if ontbreekt:
        print(f"Let op: {os.path.basename(b)} mist kolommen {ontbreekt}, opgevuld met N/A")
    for k in ontbreekt:
        d[k] = "N/A"
    frames.append(d[KOLOMMEN].replace("", "N/A"))

totaal = pd.concat(frames, ignore_index=True)
totaal = totaal.drop_duplicates(subset="hostname", keep="last") 
totaal.to_csv(os.path.join(map_met_csv, "sysinfo_all.csv"),
              sep=";", index=False, encoding="utf-8-sig")
print(f"\nSamengevoegd: {len(totaal)} computer(s) in sysinfo_all.csv")
totaal

Gevonden bestanden:
   sysinfo_Unluckydodo.csv

Samengevoegd: 1 computer(s) in sysinfo_all.csv


,hostname,model,serial_number,processor,physical_cores,cpu_max_mhz,ram_gb,fs_type,storage_gb,primary_iface,mac,ipv4,ipv6,ipv6_link_local
0,Unluckydodo,HP ProBook 455 G8 Notebook PC,5CD111NW3W,"AMD64 Family 25 Model 80 Stepping 0, AuthenticAMD",3,2301.0,7.31,NTFS,236.1,Wi-Fi,70-9C-D1-E9-BE-BB,192.168.1.38,N/A,N/A


## 7 - Valideren
Controleer of de gegevens in het CSV-bestand overeenkomen met de werkelijke configuratie van de computer. Het script vraagt de gegevens op bij Windows zelf (PowerShell/CIM), een **andere bron** dan psutil, en vergelijkt:
computernaam, serienummer, processor, aantal cores, CPU-frequentie, RAM, opslagcapaciteit, MAC-adres en IPv4-adres.
*Deze cel werkt alleen op Windows.*

In [10]:
if platform.system() != "Windows":
    print("Overgeslagen: de validatie gebruikt Windows PowerShell en werkt alleen op Windows.")
else:
    iface = filtered["primary_iface"]

    def serial_controle():
        s = _ps("(Get-WmiObject Win32_BIOS).SerialNumber")
        if s.lower() in ONGELDIGE_SERIALS:
            out = _run(["wmic", "bios", "get", "serialnumber"])
            regels = [r.strip() for r in out.splitlines()
                      if r.strip() and r.strip().lower() != "serialnumber"]
            s = regels[0] if regels else "N/A"
        return s if s.lower() not in ONGELDIGE_SERIALS else "N/A"

    echt = {
        "hostname":       os.environ.get("COMPUTERNAME", "N/A"),
        "serial_number":  serial_controle(),
        "processor":      _ps("(Get-CimInstance Win32_Processor | Select -First 1).Name"),
        "physical_cores": _ps("(Get-CimInstance Win32_Processor | Measure-Object NumberOfCores -Sum).Sum"),
        "logical_cpus":   _ps("(Get-CimInstance Win32_Processor | Measure-Object NumberOfLogicalProcessors -Sum).Sum"),
        "cpu_max_mhz":    _ps("(Get-CimInstance Win32_Processor | Select -First 1).MaxClockSpeed"),
        "ram_gb":         round(int(_ps("(Get-CimInstance Win32_ComputerSystem).TotalPhysicalMemory")) / GB, 2),
        "ram_ingebouwd_gb": round(int(_ps("(Get-CimInstance Win32_PhysicalMemory | Measure-Object Capacity -Sum).Sum")) / GB, 2),
        "storage_gb":     round(int(_ps("(Get-CimInstance Win32_LogicalDisk -Filter \"DeviceID='C:'\").Size")) / GB, 2),
        "mac":            _ps(f"(Get-NetAdapter -Name '{iface}').MacAddress").upper().replace(":", "-"),
        "ipv4":           _ps(f"(Get-NetIPAddress -InterfaceAlias '{iface}' -AddressFamily IPv4).IPAddress"),
    }

    def gelijk(a, b):
        try:
            return abs(float(a) - float(b)) < 0.5
        except ValueError:
            return str(a).strip().lower() == str(b).strip().lower()

    controles = ["hostname", "serial_number", "processor", "physical_cores",
                 "cpu_max_mhz", "ram_gb", "storage_gb", "mac", "ipv4"]

    print(f"{'veld':16} {'dataset':35} {'echt (Windows)':35} resultaat")
    for veld in controles:
        ok = gelijk(filtered[veld], echt[veld])
        print(f"{veld:16} {str(filtered[veld])[:34]:35} {str(echt[veld])[:34]:35} {'OK' if ok else 'VERSCHIL'}")

    print("\nExtra info om verschillen te verklaren:")
    print("Logische processors (Windows):", echt["logical_cpus"])
    print("Ingebouwd RAM (som van de geheugenmodules):", echt["ram_ingebouwd_gb"], "GB")

veld             dataset                             echt (Windows)                      resultaat
hostname         Unluckydodo                         UNLUCKYDODO                         OK
serial_number    5CD111NW3W                          5CD111NW3W                          OK
processor        AMD64 Family 25 Model 80 Stepping   AMD Ryzen 5 5600U with Radeon Grap  VERSCHIL
physical_cores   3                                   3                                   OK
cpu_max_mhz      2301.0                              2301                                OK
ram_gb           7.31                                7.31                                OK
storage_gb       236.1                               236.1                               OK
mac              70-9C-D1-E9-BE-BB                   70-9C-D1-E9-BE-BB                   OK
ipv4             192.168.1.38                        192.168.1.38                        OK

Extra info om verschillen te verklaren:
Logische processors (Windo

**Verschillen en mogelijke verklaringen** *(vul aan met wat jouw uitvoer toont)*

| Veld | Mogelijk verschil | Verklaring |
|---|---|---|
| RAM | Het bruikbare RAM is lager dan het ingebouwde RAM | Een deel is gereserveerd voor hardware (bv. geïntegreerde grafische kaart). We vergelijken `TotalPhysicalMemory` met de som van de modules. |
| Opslag | Kleiner dan de waarde op de verpakking (bv. 1 TB wordt ~931 GB) | Fabrikanten rekenen met 1000³, Windows en ons script met 1024³. Ook de herstelpartitie telt niet mee omdat we enkel `C:` nemen. |
| Fysieke cores | Verschil tussen psutil en Windows | Hyper-Threading: logische processors zijn niet gelijk aan fysieke cores. |
| CPU-frequentie | Kleine afwijking | psutil en Windows kunnen de basis- of maximumfrequentie anders rapporteren. |
| MAC-adres | Zelfde adres, ander formaat | Windows toont `AA-BB-CC...`, Linux `aa:bb:cc...`. Het script vergelijkt in hoofdletters met streepjes. |
| Serienummer | `N/A` bij beide | Sommige computers hebben geen echt serienummer in de BIOS (fabrieksteksten zoals "To Be Filled By O.E.M."). Dan is er niets te controleren. |

**Welke controles kunnen automatisch?** Bijna alle bovenstaande: het script vergelijkt de dataset elke keer met een tweede bron (PowerShell/CIM) en rapporteert `OK` of `VERSCHIL`. Daarnaast kunnen met Python automatisch gecontroleerd worden: of alle 14 kolommen aanwezig zijn, of er geen lege cellen zijn, of `ram_gb` en `storage_gb` getallen groter dan 0 zijn, of IP- en MAC-adressen een geldig formaat hebben (module `ipaddress` of een regex) en of de `hostname` uniek is in `sysinfo_all.csv`. Manueel blijft nodig: controleren of het serienummer overeenkomt met het label op het toestel.

## 8 - Serienummer toevoegen aan opdracht 1 t/m 7

Het serienummer zit niet in psutil. Het wordt opgevraagd met `get_serial()` (opdracht 2) en komt zo in alle volgende stappen terecht:

| Opdracht | Waar zit het serienummer? |
|---|---|
| 1 | `info["system"]["serial_number"]` (uitbreiding van de dictionary) |
| 2 | sleutel `serial_number` in `filtered` |
| 3 | zit in het `.json`-bestand |
| 4 | kolom `serial_number` in de CSV (14 kolommen) |
| 5 | kolom in het `.xlsx`-bestand, als tekst ingelezen |
| 6 | wordt samengevoegd; oude CSV's zonder die kolom krijgen `N/A` |
| 7 | gecontroleerd via `Get-WmiObject` en `wmic` |